In [13]:
%load_ext jupyter_black

The jupyter_black extension is already loaded. To reload it, use:
  %reload_ext jupyter_black


In [14]:
import warnings

warnings.filterwarnings("ignore")

In [15]:
import re
import json
import pickle
from rank_bm25 import BM25Okapi

In [16]:
# Load chunks
with open("../data/processed/chunks/trafik_kanunu.jsonl", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f]

In [17]:
# Clean the texts
def tokenize_tr(text: str) -> list[str]:
    text = text.replace("İ", "i").replace("I", "ı").lower()
    text = re.sub(r"[^a-zçğıöşü0-9\s]", " ", text)

    return text.split()

In [18]:
chunks[0]["text"]

'KARAYOLLARI TRAFİK KANUNU123\nKanun Numarası : 2918\nKabul Tarihi : 13/10/1983\nYayımlandığı Resmî Gazete : Tarih: 18/10/1983 Sayı: 18195\nYayımlandığı Düstur : Tertip: 5 Cilt: 22 Sayfa: 687\nBİRİNCİ KISIM\nGenel Esaslar\nBİRİNCİ BÖLÜM\nAmaç ve Kapsam\nAmaç:'

In [19]:
tokenize_tr(chunks[0]["text"])

['karayolları',
 'trafik',
 'kanunu123',
 'kanun',
 'numarası',
 '2918',
 'kabul',
 'tarihi',
 '13',
 '10',
 '1983',
 'yayımlandığı',
 'resm',
 'gazete',
 'tarih',
 '18',
 '10',
 '1983',
 'sayı',
 '18195',
 'yayımlandığı',
 'düstur',
 'tertip',
 '5',
 'cilt',
 '22',
 'sayfa',
 '687',
 'birinci',
 'kısım',
 'genel',
 'esaslar',
 'birinci',
 'bölüm',
 'amaç',
 've',
 'kapsam',
 'amaç']

In [20]:
tokenized_corpus = [tokenize_tr(c["text"]) for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)

In [21]:
with open("../data/processed/bm25_index.pkl", "wb") as f:
    pickle.dump({"bm25": bm25, "chunks": chunks}, f)

In [22]:
query = "kırmızı ışık"
top_k = 5

In [23]:
tokenized_query = tokenize_tr(query)
scores = bm25.get_scores(tokenized_query)
scored = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:top_k]

results = [(chunks[idx], float(score)) for idx, score in scored]

In [24]:
for chunk, score in results:
    print(f"Madde {chunk['metadata']['madde_no']} | Score: {score:.3f}")
    print(chunk["text"][:150])
    print("---")

Madde 49 | Score: 5.663
47 – Karayollarından faydalananlar aşağıdaki sıralamaya göre;
a) Trafiği düzenleme ve denetimle görevli trafik zabıtası veya özel kıyafetli veya işare
---
Madde 65 | Score: 5.190
63 – Karayolunda trafiğe çıkan bütün araçların, nicelik ve nitelikleri
yönetmelikte belirtilen şartlara uygun ışık donanımı bulundurmaları zorunludur.
---
Madde 87 | Score: 4.300
84 – Araç sürücüleri trafik kazalarında;
a) Kırmızı ışıklı trafik işaretinde veya yetkili memurun dur işaretinde geçme,
b) Taşıt giremez trafik işaret
---
Madde 66 | Score: 2.735
64 – Araçların sürülmesi sırasındaki zorunluluk ve yasaklar aşağıda gösterilmiştir.
a) Zorunluluklar:
1. (Değişik: 17/10/1996-4199 - 24 md.) Yerleşim 
---
Madde 119 | Score: 2.600
116 – (Değişik birinci fıkra: 25/6/1988 – KHK – 330/8 md.; Aynen Kabul:
31/10/1990 – 3672/7 md.) Trafiği tehlikeye düşürecek, engel olacak şekilde vey
---
